# Projeto Forecast — 03. Previsão populacional

Projeta a população de cada país para **2025** e **2030** a partir da base limpa (`02_limpeza.ipynb`).

Roteiro:
1. **Backtest**: treinar só com dados até 2015 e prever 2020 e 2022, que já conhecemos, para escolher o método.
2. **Previsão final**: aplicar o método escolhido aos dados até 2022.
3. **Exportar** o resultado em `data/forecast_populacao.csv`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path('data')
df = pd.read_csv(DATA / 'world_population_limpo.csv')

ANOS = [1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022]
pop = df.set_index('pais')[[f'pop_{a}' for a in ANOS]]
pop.columns = ANOS
pop.head()

,1970,1980,1990,2000,2010,2015,2020,2022
pais,,,,,,,,
Afghanistan,10752971,12486631,10694796,19542982,28189672,33753499,38972230,41128771
Albania,2324731,2941651,3295066,3182021,2913399,2882481,2866849,2842321
Algeria,13795915,18739378,25518074,30774621,35856344,39543154,43451666,44903225
American Samoa,27075,32886,47818,58230,54849,51368,46189,44273
Andorra,19860,35611,53569,66097,71519,71746,77700,79824


## 1. Métodos candidatos

Os anos não são igualmente espaçados (de 10 em 10 até 2010, depois 2015, 2020 e 2022), então usamos métodos que trabalham com o ano real de cada ponto:

- **ingênuo**: a população fica igual à última observada (referência mínima: um bom método tem que ganhar dele);
- **linear**: reta ajustada aos últimos `n` pontos;
- **exponencial**: crescimento a taxa constante (reta sobre o log da população) nos últimos `n` pontos.

In [2]:
def prever(pop, ano_alvo, metodo, n=2):
    """Prevê a população de cada país em ano_alvo usando os últimos n anos de pop."""
    anos = np.array(pop.columns[-n:], dtype=float)
    valores = pop.iloc[:, -n:].to_numpy(dtype=float)
    if metodo == 'ingenuo':
        return pd.Series(valores[:, -1], index=pop.index)
    if metodo == 'exponencial':
        valores = np.log(valores)
    # polyfit aceita várias séries de uma vez: uma coluna por país
    inclinacao, intercepto = np.polyfit(anos, valores.T, 1)
    previsto = inclinacao * ano_alvo + intercepto
    if metodo == 'exponencial':
        previsto = np.exp(previsto)
    return pd.Series(previsto, index=pop.index)

## 2. Backtest: treino até 2015, teste em 2020 e 2022

Métrica: **erro percentual absoluto** de cada país (|previsto − real| ÷ real). Olhamos a mediana (país típico), a média e o percentil 90 (os 10% piores casos).

In [3]:
treino = pop[[a for a in ANOS if a <= 2015]]

candidatos = [('ingenuo', 1), ('linear', 2), ('exponencial', 2), ('linear', 3), ('exponencial', 3), ('linear', 4)]
linhas = []
for metodo, n in candidatos:
    for ano in (2020, 2022):
        erro = (prever(treino, ano, metodo, n) - pop[ano]).abs() / pop[ano] * 100
        linhas.append({'metodo': metodo, 'n_pontos': n, 'ano_teste': ano,
                       'erro_mediano_%': erro.median(), 'erro_medio_%': erro.mean(),
                       'erro_p90_%': erro.quantile(0.9)})

backtest = pd.DataFrame(linhas).round(2)
backtest.pivot_table(index=['metodo', 'n_pontos'], columns='ano_teste',
                     values=['erro_mediano_%', 'erro_p90_%']).sort_values(('erro_mediano_%', 2022))

erro_mediano_%       erro_p90_%       
ano_teste                      2020  2022       2020   2022
metodo      n_pontos                                       
linear      2                  1.04  1.75       4.90   7.16
exponencial 2                  1.07  1.87       4.95   7.88
            3                  1.84  2.77       7.29   9.92
linear      3                  2.07  2.91       6.39   8.31
            4                  2.69  3.45       8.89  10.53
ingenuo     1                  6.02  7.96      12.63  16.82

**Conclusão do backtest:** a **reta entre os dois últimos pontos** tem o menor erro (mediana ~1% em 2020 e ~1,75% em 2022) e ganha com folga do método ingênuo. Usar mais pontos piora, porque mistura tendências antigas (décadas de 1990/2000) com a dinâmica recente. Linear e exponencial com 2 pontos ficam praticamente empatados; ficamos com o **linear**, que teve erro um pouco menor.

### Onde o modelo mais erra (teste em 2022)

In [4]:
prev_2022 = prever(treino, 2022, 'linear', 2)
erros = pd.DataFrame({'real_2022': pop[2022], 'previsto_2022': prev_2022.round()})
erros['erro_%'] = ((erros['previsto_2022'] - erros['real_2022']) / erros['real_2022'] * 100).round(1)
erros.reindex(erros['erro_%'].abs().sort_values(ascending=False).index).head(10)

,real_2022,previsto_2022,erro_%
pais,,,
Lebanon,5489739,8363336.0,52.3
Syria,22125249,14819839.0,-33.0
Oman,4576298,6025583.0,31.7
Qatar,2695122,3396070.0,26.0
Kuwait,4268873,5260285.0,23.2
South Sudan,10913164,13266131.0,21.6
Montserrat,4390,5228.0,19.1
Moldova,3272996,2716271.0,-17.0
Venezuela,28301696,33070288.0,16.8


Os maiores erros são países com rupturas que nenhuma tendência capta (guerras, crises migratórias, mudanças bruscas de fluxo de trabalhadores estrangeiros). É uma limitação conhecida de modelos baseados só no histórico.

## 3. Previsão final: 2025 e 2030

No backtest os dois últimos pontos estavam a 5 anos de distância (2010 e 2015). Para manter um intervalo parecido e não depender só de 2020→2022 (período afetado pela pandemia), a reta final usa **2015 e 2022**.

In [5]:
base = pop[[2015, 2022]]

forecast = pd.DataFrame({
    'cca3': df.set_index('pais')['cca3'],
    'continente': df.set_index('pais')['continente'],
    'pop_2022': pop[2022],
    'pop_2025': prever(base, 2025, 'linear', 2).round().astype('int64'),
    'pop_2030': prever(base, 2030, 'linear', 2).round().astype('int64'),
})
forecast['var_2022_2030_%'] = ((forecast['pop_2030'] / forecast['pop_2022'] - 1) * 100).round(2)

assert (forecast[['pop_2025', 'pop_2030']] > 0).all().all()
forecast.sort_values('pop_2030', ascending=False).head(10)

,cca3,continente,pop_2022,pop_2025,pop_2030,var_2022_2030_%
pais,,,,,,
India,IND,Asia,1417173173,1457590316,1524952222,7.61
China,CHN,Asia,1425887337,1439675289,1462655210,2.58
United States,USA,North America,338289857,344153606,353926521,4.62
Indonesia,IDN,Asia,275501339,282533926,294254904,6.81
Pakistan,PAK,Asia,235824862,246477247,264231221,12.05
Nigeria,NGA,Africa,218541212,233346395,258021700,18.07
Brazil,BRA,South America,215313498,219652909,226885261,5.37
Bangladesh,BGD,Asia,171186372,176910531,186450797,8.92
Ethiopia,ETH,Africa,123379924,132340508,147274814,19.37


### Totais por continente e mundo

In [6]:
por_continente = forecast.groupby('continente')[['pop_2022', 'pop_2025', 'pop_2030']].sum()
por_continente.loc['Mundo'] = por_continente.sum()
(por_continente / 1e6).round(1).rename(columns=lambda c: c + ' (mi)')

,pop_2022 (mi),pop_2025 (mi),pop_2030 (mi)
continente,,,
Africa,1426.7,1523.4,1684.6
Asia,4721.4,4834.2,5022.1
Europe,743.1,743.8,745.0
North America,600.3,613.1,634.5
Oceania,45.0,47.0,50.3
South America,436.8,447.0,463.9
Mundo,7973.4,8208.5,8600.4


### Maiores crescimentos e quedas previstos até 2030

In [7]:
cols = ['continente', 'pop_2022', 'pop_2030', 'var_2022_2030_%']
display(forecast.nlargest(5, 'var_2022_2030_%')[cols])
display(forecast.nsmallest(5, 'var_2022_2030_%')[cols])

,continente,pop_2022,pop_2030,var_2022_2030_%
pais,,,,
Mayotte,Africa,326101,413594,26.83
Niger,Africa,26207977,33156380,26.51
Tokelau,Oceania,1871,2348,25.49
Somalia,Africa,17597511,21978774,24.90
Angola,Africa,35588987,44116148,23.96


,continente,pop_2022,pop_2030,var_2022_2030_%
pais,,,,
Marshall Islands,Oceania,41569,32608,-21.56
Lebanon,Asia,5489739,4450652,-18.93
American Samoa,Oceania,44273,36164,-18.32
Montserrat,North America,4390,3625,-17.43
Ukraine,Europe,39701739,33666510,-15.20


## 4. Exportação

In [8]:
forecast.reset_index().to_csv(DATA / 'forecast_populacao.csv', index=False)
print('Salvo:', DATA / 'forecast_populacao.csv', forecast.shape)

Salvo: data\forecast_populacao.csv (234, 6)


## Limitações

- Só 8 observações por país e espaçadas de forma irregular: não há como captar sazonalidade ou ciclos.
- A tendência linear supõe que o ritmo de 2015–2022 continua; rupturas (guerras, migração, políticas de natalidade) não são previstas.
- Para horizontes longos (2050+), projeções demográficas usam natalidade, mortalidade e migração (como as da ONU), não apenas a tendência.